# SynthID 워터마크 논문 읽기

토너먼트 워터마크의 분포 보존·탐지·편집 내성·엔트로피·속도를 작은 언어 모델로 확인합니다.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/No1Joon/oh-my-notebooks/blob/main/notebooks/synthid_tournament_watermark.ipynb)

[논문: Scalable watermarking for identifying large language model outputs](https://doi.org/10.1038/s41586-024-08025-4)

**T4 GPU 런타임 권장**: Colab의 `런타임 → 런타임 유형 변경 → T4 GPU`를 선택하고 위에서부터 실행하세요. GPU가 없으면 CPU를 사용하며 시간이 오래 걸립니다.
이 노트북은 논문의 아이디어를 이해하기 위한 교육용 구현이며 공식 SynthID 구현은 아닙니다.
글의 생성 실험(각 500편, 400토큰, M3 Pro)보다 규모를 줄였으므로 수치가 다릅니다. 실험 1은 모델 없이 원본과 같은 키 20만 개를 사용하고, 실험 3(m sweep)은 생략합니다.

## 준비

Colab 기본 torch를 그대로 사용합니다. transformers가 없거나 4.46 미만일 때만 설치하며, 모델 가중치는 뒤의 생성 함수 셀에서 내려받습니다.

In [ ]:
import importlib.metadata
import subprocess
import sys

try:
    version = importlib.metadata.version("transformers")
    needs_install = tuple(int(v) for v in version.split(".")[:2]) < (4, 46)
except importlib.metadata.PackageNotFoundError:
    needs_install = True
if needs_install:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "transformers>=4.46"])

In [ ]:
import os
import time
import statistics
import torch
import transformers

os.environ["TOKENIZERS_PARALLELISM"] = "false"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DTYPE = torch.float16 if DEVICE.type == "cuda" else torch.float32
print("torch:", torch.__version__)
print("transformers:", transformers.__version__)
print("device:", DEVICE, "dtype:", DTYPE)
if DEVICE.type == "cuda":
    print("GPU:", torch.cuda.get_device_name())

## 워터마크 함수

응답의 직전 네 토큰으로 키를 만들고, 층별 이진 함수로 2자 토너먼트의 확률 분포를 갱신합니다. 반복 문맥에서는 워터마크를 적용하지 않으며 탐지에서도 그 위치를 제외합니다. `DEFAULT_KEY`는 누구나 재현할 수 있도록 공개한 실험용 상수입니다.

In [ ]:
"""Tournament and Gumbel watermarks. Hashing/scoring run on CPU int64."""

import math
from dataclasses import dataclass, field

import torch

H = 4
DEFAULT_KEY = 20241107
MASK63 = (1 << 63) - 1
SCHEMA_VERSION = 1

# Limit tiny tensor operations' thread overhead; also used by experiment scripts.
torch.set_num_threads(min(4, torch.get_num_threads()))


def mix64(x: torch.Tensor) -> torch.Tensor:
    """SplitMix64 finalizer; signed overflow and logical shifts, modulo 2**64.

    Constants are the signed representations of the standard unsigned constants.
    No Python hash(), device-dependent float hash, or global RNG state is used.
    """
    x = x.to(device="cpu", dtype=torch.int64)
    x = (x ^ ((x >> 30) & ((1 << 34) - 1))) * -4658895280553007687
    x = (x ^ ((x >> 27) & ((1 << 37) - 1))) * -7723592293110705685
    return x ^ ((x >> 31) & ((1 << 33) - 1))


def context_seed(context: torch.Tensor, key: int | torch.Tensor) -> torch.Tensor:
    """context: [..., <=4] response token IDs; empty context is supported."""
    context = torch.as_tensor(context, dtype=torch.int64, device="cpu")[..., -H:]
    key = torch.as_tensor(key, dtype=torch.int64, device="cpu")
    seed = mix64(key ^ (context.shape[-1] + 0x12345678))
    seed = seed.expand(torch.broadcast_shapes(seed.shape, context.shape[:-1]))
    for token in context.unbind(-1):
        seed = mix64(seed ^ mix64(token + 0x6A09E667))
    return seed


def g_values(seeds: torch.Tensor, tokens: torch.Tensor, m: int = 30) -> torch.Tensor:
    """Return float32 g of shape [..., m, K], for seeds [...] and tokens [..., K]."""
    if m < 1:
        raise ValueError("m must be positive")
    seeds = torch.as_tensor(seeds, dtype=torch.int64, device="cpu")
    tokens = torch.as_tensor(tokens, dtype=torch.int64, device="cpu")
    layers = torch.arange(1, m + 1, dtype=torch.int64)
    layer_hash = mix64(layers + 0x243F6A88)
    hashed = mix64(seeds[..., None, None] ^ layer_hash[:, None]
                   ^ mix64(tokens[..., None, :] + 0x13198A2E))
    return (hashed & 1).to(torch.float32)


def uniform_values(seeds: torch.Tensor, tokens: torch.Tensor) -> torch.Tensor:
    """Domain-separated, open-interval uniforms, float64 (53 random bits)."""
    seeds = torch.as_tensor(seeds, dtype=torch.int64, device="cpu")
    tokens = torch.as_tensor(tokens, dtype=torch.int64, device="cpu")
    bits = mix64(seeds[..., None] ^ mix64(tokens + 0x3C6EF372) ^ 0x5A827999)
    # 52 bits plus half a unit is strictly inside (0, 1), even after rounding.
    return (((bits >> 12) & ((1 << 52) - 1)).to(torch.float64) + 0.5) / 2**52


def tournament_distribution(p: torch.Tensor, g: torch.Tensor,
                            return_history: bool = False):
    """Exact two-way tournament distribution, with one bit function per layer."""
    p = p.to(device="cpu", dtype=torch.float32).clone()
    history = [p.clone()] if return_history else None
    for layer in g.unbind(-2):
        mean_g = (p * layer).sum(-1, keepdim=True)
        p = p * (1.0 + layer - mean_g)
        # Only correct float32 roundoff; the analytic update already sums to one.
        p = p.clamp_min(0)
        p = p / p.sum(-1, keepdim=True)
        if history is not None:
            history.append(p.clone())
    return (p, torch.stack(history, -2)) if return_history else p


def gumbel_select(p: torch.Tensor, seeds: torch.Tensor,
                  tokens: torch.Tensor) -> torch.Tensor:
    """Index of argmax r_v**(1/p_v), evaluated stably as log(r_v)/p_v."""
    p = p.to(device="cpu", dtype=torch.float32)
    r = uniform_values(seeds, tokens)
    objective = torch.where(p > 0, r.log() / p.to(torch.float64), -torch.inf)
    return objective.argmax(-1)


@dataclass
class ContextTracker:
    """One tracker per response; True means repeated context, hence masked."""
    seen: set[tuple[int, ...]] = field(default_factory=set)

    def observe(self, context) -> bool:
        context = tuple(int(v) for v in context[-H:])
        repeated = context in self.seen
        self.seen.add(context)
        return repeated


def sequence_contexts(tokens: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
    """Return context seeds' inputs (padded) and repeated-context mask per row.

    Padded contexts are an internal convenience; lengths must be honored by caller.
    """
    tokens = torch.as_tensor(tokens, dtype=torch.int64, device="cpu")
    if tokens.ndim == 1:
        tokens = tokens[None, :]
    n, length = tokens.shape
    contexts = torch.zeros((n, length, H), dtype=torch.int64)
    masked = torch.zeros((n, length), dtype=torch.bool)
    for lag in range(1, min(H, length) + 1):
        contexts[:, lag:, H-lag] = tokens[:, :-lag]
    for row, sequence in enumerate(tokens.tolist()):
        tracker = ContextTracker()
        masked[row] = torch.tensor([
            tracker.observe(sequence[max(0, t-H):t]) for t in range(length)
        ], dtype=torch.bool)
    return contexts, masked


def detection_values(tokens: torch.Tensor, key: int = DEFAULT_KEY, m: int = 30,
                     method: str = "synthid") -> tuple[torch.Tensor, torch.Tensor]:
    """Recompute masking and g/r from tokens alone, including after edits.

    values is [n, length, m] for SynthID, [n, length, 1] for Gumbel.
    """
    tokens = torch.as_tensor(tokens, dtype=torch.int64, device="cpu")
    if tokens.ndim == 1:
        tokens = tokens[None, :]
    contexts, masked = sequence_contexts(tokens)
    seeds = torch.empty_like(tokens)
    for t in range(min(H, tokens.shape[1])):
        seeds[:, t] = context_seed(contexts[:, t, H-t:] if t else contexts[:, t, :0], key)
    if tokens.shape[1] > H:
        seeds[:, H:] = context_seed(contexts[:, H:, :], key)
    if method == "synthid":
        values = g_values(seeds, tokens[..., None], m).squeeze(-1)
    elif method == "gumbel":
        values = -torch.log1p(-uniform_values(seeds, tokens[..., None]))
    else:
        raise ValueError(f"Unknown detection method: {method}")
    return values, masked


def score_tokens(tokens: torch.Tensor, key: int = DEFAULT_KEY, m: int = 30,
                 method: str = "synthid") -> tuple[torch.Tensor, torch.Tensor]:
    values, masked = detection_values(tokens, key, m, method)
    valid = ~masked
    count = valid.sum(-1)
    scores = (values.to(torch.float64) * valid[..., None]).sum((-1, -2))
    scores /= count.clamp_min(1) * values.shape[-1]
    scores[count == 0] = torch.nan
    return scores, count


def synthid_score(tokens, key=DEFAULT_KEY, m=30):
    return score_tokens(tokens, key, m, "synthid")[0]


def gumbel_score(tokens, key=DEFAULT_KEY):
    return score_tokens(tokens, key, method="gumbel")[0]

## 실험 1. 분포 보존

독립적인 키 20만 개에 대해 원래 분포와 확률 갱신·실제 2자 토너먼트의 빈도를 비교하고, 3자 토너먼트에서 생기는 왜곡도 확인합니다. 이는 **키에 대해 평균한 분포**의 성질이며 고정된 키의 분포가 매번 원래와 같다는 뜻은 아닙니다. 이 실험은 원본과 같은 조건이고, 뒤의 생성 실험은 글의 각 500편·400토큰·M3 Pro 조건보다 축소하여 값이 다릅니다.

In [ ]:
def actual_tournament(p, g, arity, rng):
    """Draw arity**m independent leaves; maximize g, break ties uniformly."""
    n, m, _ = g.shape
    leaves = torch.multinomial(p.expand(n, -1), arity**m, replacement=True, generator=rng)
    rows = torch.arange(n)[:, None, None]
    for layer in range(m):
        groups = leaves.reshape(n, -1, arity)
        bits = g[:, layer, :][rows, groups]
        # IID continuous tie priorities give uniform choice among tied maxima.
        priority = bits + torch.rand(groups.shape, generator=rng)
        winner = priority.argmax(-1, keepdim=True)
        leaves = groups.gather(-1, winner).squeeze(-1)
    return leaves[:, 0]


rng = torch.Generator().manual_seed(2024)
p = torch.tensor([0.50, 0.30, 0.15, 0.05])
keys = torch.randint(0, 2**62, (200000,), generator=rng)
seeds = context_seed(torch.tensor([1, 2, 3, 4]), keys)
g = g_values(seeds, torch.arange(4), 3)
exact = tournament_distribution(p.expand(200000, -1), g)
selected = torch.multinomial(exact, 1, generator=rng).squeeze(-1)
two = actual_tournament(p, g, 2, rng)
three = actual_tournament(p, g, 3, rng)
freq = [torch.bincount(x, minlength=4).float() / 200000 for x in (selected, two, three)]
print(f"exp1: independent random keys={200000}, m=3; seed=2024")
print("token       original    update draw    8-leaf 2-way    27-leaf 3-way   MC SE(null)")
for i, name in enumerate(["망고", "리치", "파파야", "두리안"]):
    se = (p[i] * (1-p[i]) / 200000).sqrt().item()
    print(f"{name:8s} {p[i]:.6f}    {freq[0][i]:.6f}       {freq[1][i]:.6f}       {freq[2][i]:.6f}      {se:.6f}")
print("Key-averaged analytic distribution:", [round(v, 6) for v in exact.mean(0).tolist()])
deviations = [float((f - p).abs().max()) for f in freq]
print("Maximum absolute errors (update / 2-way / 3-way):", deviations)
print("Paired update vs 2-way maximum frequency gap:", float((freq[0]-freq[1]).abs().max()))
print("3-way systematically favors lower-probability tokens under random binary g.")
fixed_g = g_values(context_seed(torch.tensor([1, 2, 3, 4]), 17), torch.arange(4), 3)
_, history = tournament_distribution(p, fixed_g, return_history=True)
print("Fixed key=17; g rows (layer 1..3):", fixed_g.int().tolist())
for layer, distribution in enumerate(history):
    print(f"layer {layer}: {distribution.tolist()}")
if 200000 >= 200000:
    assert max(deviations[:2]) < 0.005, "Distribution preservation check failed"
    assert float((freq[0]-freq[1]).abs().max()) < 0.005
    print("PASS: both two-way methods within 0.005 of original and each other")

## 생성 함수와 탐지 통계

모델은 Qwen2.5-0.5B-Instruct를 사용합니다. 질문은 직접 작성한 설명형(`eli5`)·나열형(`list`) 지시이며 외부 ELI5 데이터셋은 사용하지 않습니다.
온도 0.7, top-k 50, 응답 토큰만을 이용한 문맥, 특수 토큰 제외와 KV 캐시를 원본처럼 유지하며 EOS에서도 조기 종료하지 않고 지정 길이를 생성합니다.
FPR 1%는 none 점수의 선형 99백분위수로 보정한 **목표치**로, 같은 none 표본의 관측 FPR도 출력합니다. 별도 검증 표본의 FPR 보장은 아니며 작은 표본·동점 때문에 관측치가 1%와 다를 수 있습니다.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

# 모델 출처: https://huggingface.co/Qwen/Qwen2.5-0.5B-Instruct
# 모델 가중치 및 토크나이저 라이선스: Apache-2.0. 실행 시 다운로드합니다.
# 아래 질문 목록은 직접 작성한 예제이며 외부 데이터셋을 내려받지 않습니다.
MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"
TEMPERATURE = 0.7
TOP_K = 50

# Handwritten questions, not fetched from a benchmark or generated by an API.
ELI5_TOPICS = [
    "why is the sky blue?", "how do clouds make rain?",
    "why does the Moon change shape?", "what makes thunder?",
    "why do rainbows have colors?", "how does snow form?",
    "why is the ocean salty?", "how do waves form?",
    "why do rivers flow downhill?", "what makes volcanoes erupt?",
    "why do earthquakes happen?", "how do mountains form?",
    "why is sand made of tiny grains?", "how do caves form?",
    "why do seasons change?", "why is it colder at the poles?",
    "how does the Sun make light?", "why do stars twinkle?",
    "what keeps planets in orbit?", "why does Earth spin?",
    "what is gravity?", "why do astronauts float?",
    "how does a rocket move?", "what is a comet?",
    "why does the Moon have craters?", "what is a galaxy?",
    "how do plants grow from seeds?", "why are leaves green?",
    "why do leaves fall in autumn?", "how do flowers attract bees?",
    "why do plants need sunlight?", "how do roots take up water?",
    "how do mushrooms grow?", "why do cacti have spines?",
    "how do trees make rings?", "why do some fruits taste sweet?",
    "how do butterflies change from caterpillars?", "why do birds have feathers?",
    "how do fish breathe underwater?", "why do dogs wag their tails?",
    "how do cats see in the dark?", "why do bears hibernate?",
    "how do ants find food?", "why do spiders make webs?",
    "how do bats find their way?", "why do whales come up for air?",
    "how do penguins stay warm?", "why do snakes shed their skin?",
    "how do frogs change as they grow?", "why do fireflies glow?",
    "why does your heart beat?", "how do lungs work?",
    "why do we need sleep?", "how does the brain remember things?",
    "why do we get hungry?", "how do bones heal?",
    "why do we sweat?", "why do we shiver?",
    "how do ears hear sounds?", "how do eyes see colors?",
    "why do we have fingerprints?", "how does soap clean hands?",
    "why do teeth need brushing?", "how do vaccines teach the body?",
    "why does ice float?", "how does water boil?",
    "why does ice melt?", "how does a refrigerator keep food cold?",
    "why does metal feel colder than wood?", "how does a thermometer work?",
    "what is electricity?", "how does a battery work?",
    "why do magnets attract some metals?", "how does a light bulb work?",
    "how does a solar panel make electricity?", "how does a wind turbine work?",
    "how does a bicycle stay balanced?", "how do airplane wings lift a plane?",
    "why do boats float?", "how do submarines go underwater?",
    "how does a car engine work?", "why do brakes slow a bicycle?",
    "how does an elevator move?", "how does a bridge hold weight?",
    "how does a computer follow instructions?", "how does the internet send messages?",
    "how does a camera take pictures?", "how does a microphone record sound?",
    "why does an echo happen?", "how do musical instruments make notes?",
    "why does bread rise?", "how does popcorn pop?",
    "why does sugar dissolve in water?", "why do apples turn brown?",
    "how does a clock keep time?", "why do shadows change during the day?",
    "how does a mirror reflect light?", "why does a straw look bent in water?",
    "how does a rainbow form in a soap bubble?", "why does rubbing hands make them warm?",
]
ELI5_PROMPTS = ["Explain like I'm five: " + topic for topic in ELI5_TOPICS]
LIST_PROMPTS = [
    "List the integers from 1 to 300, separated by commas.",
    "List the integers from 300 down to 1, separated by commas.",
    "List the even integers from 2 to 600, separated by commas.",
    "List the odd integers from 1 to 599, separated by commas.",
    "Write the lowercase English alphabet in order, then repeat it 30 times.",
    "Write the uppercase English alphabet in order, then repeat it 30 times.",
    "Repeat the exact phrase 'one two three four five' 100 times.",
    "Repeat the exact word 'hello' 300 times, separated by spaces.",
    "Repeat the sequence 'A B C D E F G' 100 times.",
    "List the months January through December in order, repeating 30 times.",
    "List the weekdays Monday through Sunday in order, repeating 50 times.",
    "Count from 1 to 300, writing one number on each line.",
    "Write the multiplication table for 2 from 2 times 1 through 2 times 100.",
    "Write the multiplication table for 10 from 10 times 1 through 10 times 100.",
    "List the multiples of 5 from 5 to 1500, separated by commas.",
    "Repeat 'true false' 200 times, separated by spaces.",
    "Repeat '0 1 0 1 0 1 0 1' 100 times.",
    "List the integers from 1000 to 1300 in ascending order.",
    "Repeat the exact sentence 'The cat sat on the mat.' 100 times.",
    "Output a JSON array containing the integers from 1 to 300 in order.",
]
assert len(ELI5_PROMPTS) == 100 and len(LIST_PROMPTS) == 20


def synchronize(device):
    if torch.device(device).type == "cuda":
        torch.cuda.synchronize(device)


class GenerationEngine:
    def __init__(self):
        self.device = DEVICE
        dtype = DTYPE
        started = time.perf_counter()
        self.tokenizer = AutoTokenizer.from_pretrained(
            MODEL_ID, padding_side="left")
        if self.tokenizer.pad_token_id is None:
            self.tokenizer.pad_token = self.tokenizer.eos_token
        self.model = AutoModelForCausalLM.from_pretrained(
            MODEL_ID, torch_dtype=dtype, attn_implementation="sdpa"
        ).to(self.device).eval()
        blocked = set(self.tokenizer.all_special_ids)
        for value in (self.model.config.eos_token_id,
                      self.model.generation_config.eos_token_id):
            if value is not None:
                blocked.update(value if isinstance(value, list) else [value])
        self.blocked = sorted(blocked)
        self.allowed_token_ids = torch.tensor(
            [v for v in range(self.model.config.vocab_size) if v not in blocked],
            dtype=torch.int64)
        synchronize(self.device)
        self.load_seconds = time.perf_counter() - started

    @torch.inference_mode()
    def generate(self, method="synthid", m=30, n=200, length=200, prompts="eli5",
                 batch=8, seed=1234, key=DEFAULT_KEY, save_entropy=False,
                 progress=True):
        if method not in {"none", "synthid", "gumbel"}:
            raise ValueError("method must be none, synthid or gumbel")
        if min(n, length, batch, m) < 1:
            raise ValueError("n, length, batch, m must be positive")
        prompt_set = ELI5_PROMPTS if prompts == "eli5" else LIST_PROMPTS
        rng = torch.Generator(device="cpu").manual_seed(seed)
        all_tokens = torch.empty((n, length), dtype=torch.int64)
        all_masks = torch.empty((n, length), dtype=torch.bool)
        all_entropy = torch.empty((n, length), dtype=torch.float32) if save_entropy else None
        synchronize(self.device)
        started = time.perf_counter()
        for start in range(0, n, batch):
            size = min(batch, n-start)
            texts = [self.tokenizer.apply_chat_template(
                [{"role": "user", "content": prompt_set[(start+i) % len(prompt_set)]}],
                tokenize=False, add_generation_prompt=True) for i in range(size)]
            inputs = self.tokenizer(texts, padding=True, return_tensors="pt",
                                    add_special_tokens=False).to(self.device)
            attention_mask = inputs["attention_mask"]
            input_ids = inputs["input_ids"]
            cache = None
            output = all_tokens[start:start+size]
            masks = all_masks[start:start+size]
            trackers = [ContextTracker() for _ in range(size)]
            for t in range(length):
                positions = attention_mask.long().cumsum(-1) - 1
                positions.masked_fill_(attention_mask == 0, 0)
                if cache is not None:
                    positions = positions[:, -1:]
                result = self.model(input_ids=input_ids, attention_mask=attention_mask,
                                    position_ids=positions, past_key_values=cache,
                                    use_cache=True, num_logits_to_keep=1)
                cache = result.past_key_values
                logits = result.logits[:, -1, :].float() / TEMPERATURE
                logits[:, self.blocked] = -torch.inf
                top_logits, candidate_ids = torch.topk(logits, TOP_K, dim=-1)
                # Float32 probabilities on both CPU and CUDA; only K values copied.
                p = torch.softmax(top_logits, dim=-1).cpu()
                candidate_ids = candidate_ids.cpu()
                if save_entropy:
                    all_entropy[start:start+size, t] = -(p * p.clamp_min(1e-30).log()).sum(-1)
                context = output[:, max(0, t-4):t]
                seeds = context_seed(context, key)
                masks[:, t] = torch.tensor([trackers[i].observe(context[i].tolist())
                                           for i in range(size)])
                if method == "synthid":
                    watermarked = tournament_distribution(p, g_values(seeds, candidate_ids, m))
                    distribution = torch.where(masks[:, t, None], p, watermarked)
                    choice = torch.multinomial(distribution, 1, generator=rng).squeeze(-1)
                elif method == "gumbel":
                    choice = gumbel_select(p, seeds, candidate_ids)
                    if masks[:, t].any():
                        rows = masks[:, t]
                        choice[rows] = torch.multinomial(p[rows], 1, generator=rng).squeeze(-1)
                else:
                    choice = torch.multinomial(p, 1, generator=rng).squeeze(-1)
                sampled = candidate_ids.gather(-1, choice[:, None]).squeeze(-1)
                output[:, t] = sampled
                input_ids = sampled[:, None].to(self.device)
                attention_mask = torch.cat((attention_mask, torch.ones(
                    (size, 1), dtype=attention_mask.dtype, device=self.device)), dim=-1)
                del result, logits
            if progress:
                print(f"  {method} m={m} {prompts}: {start+size}/{n} x {length}", flush=True)
        synchronize(self.device)
        elapsed = time.perf_counter() - started
        data = dict(schema_version=SCHEMA_VERSION, tokens=all_tokens, masked=all_masks,
                    method=method, m=m, key=key, prompts=prompts, device=str(self.device),
                    total_seconds=elapsed, model_load_seconds=self.load_seconds,
                    seed=seed, batch=batch, n=n, length=length, model=MODEL_ID,
                    temperature=TEMPERATURE, top_k=TOP_K, context_source="response_only",
                    vocab_size=self.model.config.vocab_size, blocked_token_ids=self.blocked,
                    allowed_token_ids=self.allowed_token_ids,
                    prompt_indices=torch.arange(n) % len(prompt_set))
        if save_entropy:
            data["entropy"] = all_entropy
        return data



engine = GenerationEngine()
print("Model loaded:", MODEL_ID, engine.device)

In [ ]:
def ensure_comparable(baseline, other):
    for name in ("key", "model", "temperature", "top_k", "context_source", "prompts"):
        if baseline[name] != other[name]:
            raise ValueError(f"Incompatible {name}: {baseline[name]} vs {other[name]}")


def metrics(null_scores, positive_scores):
    """Strictly above empirical 99th percentile; ties receive half credit in AUC."""
    null_scores, positive_scores = null_scores.double(), positive_scores.double()
    if not torch.isfinite(null_scores).all() or not torch.isfinite(positive_scores).all():
        raise ValueError("Non-finite scores (possibly no unmasked positions)")
    threshold = torch.quantile(null_scores, 0.99)
    fpr = (null_scores > threshold).double().mean()
    tpr = (positive_scores > threshold).double().mean()
    difference = positive_scores[:, None] - null_scores[None, :]
    auc = ((difference > 0).double() + 0.5 * (difference == 0)).mean()
    return dict(threshold=float(threshold), fpr=float(fpr), tpr=float(tpr), auc=float(auc))


def print_metrics(label, null_scores, positive_scores):
    result = metrics(null_scores, positive_scores)
    print(f"{label}: threshold={result['threshold']:.6f}, FPR={result['fpr']:.4f}, "
          f"TPR={result['tpr']:.4f}, AUC={result['auc']:.6f}")
    return result


def score_data(data, method, length, m=30, n=None):
    return score_tokens(data["tokens"][:n, :length], data["key"], m, method)



# 생성 결과는 메모리에서 공유합니다. 파일 저장이나 원본 프로젝트 접근은 없습니다.
datasets = {}
def load_data(directory, method, m=30, prompts="eli5", n=None, length=None):
    data = datasets[(method, prompts)]
    assert data["m"] == m
    assert n is None or data["tokens"].shape[0] >= n
    assert length is None or data["tokens"].shape[1] >= length
    return data

def print_scope(quick=False):
    print("threshold=linear 99th percentile of the same none pool; score > threshold")
    print("Target FPR=1%; empirical calibration, no held-out FPR estimate.")

N_DETECT = 200
LENGTH = 200
LENGTHS = [50, 100, 200]
N_ENTROPY = 50
BATCH = 8
LATENCY_LENGTH = 200

## 실험 2. 길이별 탐지

none·SynthID(m=30)·Gumbel을 각각 200편, 200토큰 생성하고 앞의 50·100·200토큰에서 목표 FPR 1%의 TPR을 비교합니다. 글의 각 500편·400토큰·M3 Pro보다 축소된 조건이므로 수치가 다르며, 마지막 층별 평균은 어느 층에서 신호가 생기는지 보여줍니다.

In [ ]:
for method in ("none", "synthid", "gumbel"):
    datasets[(method, "eli5")] = engine.generate(
        method=method, m=30, n=N_DETECT, length=LENGTH, batch=BATCH)
n, lengths, directory = N_DETECT, LENGTHS, None
print_scope(False)
baseline = load_data(directory, "none", n=n, length=max(lengths))
for method in ("synthid", "gumbel"):
    watermarked = load_data(directory, method, n=n, length=max(lengths))
    ensure_comparable(baseline, watermarked)
    print(f"\n{method}, n={n}, m=30")
    for length in lengths:
        null, counts = score_data(baseline, method, length, n=n)
        positive, _ = score_data(watermarked, method, length, n=n)
        print_metrics(f"length={length}", null, positive)
        theory_mean = 0.5 if method == "synthid" else 1.0
        # Mixture of per-response null variances with actual unmasked counts.
        theory_var = 0.25 / (30 * counts.double()) if method == "synthid" else 1 / counts.double()
        theory_sd = theory_var.mean().sqrt().item()
        print(f"  none mean={null.mean():.6f}, sample SD={null.std():.6f}; "
              f"independent-null mean={theory_mean:.3f}, SD={theory_sd:.6f}; "
              f"unmasked count mean={counts.double().mean():.2f}")
        print(f"  watermarked mean={positive.mean():.6f}, sample SD={positive.std():.6f}")
synthid = load_data(directory, "synthid", n=n, length=max(lengths))
values, masked = detection_values(synthid["tokens"][:n, :max(lengths)], synthid["key"], 30)
means = (values * (~masked)[..., None]).sum((0, 1)) / (~masked).sum()
print(f"\nSynthID per-layer token-weighted means (length={max(lengths)}):")
for layer, mean in enumerate(means.tolist(), 1):
    print(f"layer {layer:2d}: {mean:.6f}")
print("Independent-bit SD is a reference, not a guaranteed null distribution: "
      "shared/repeated contexts and finite samples can introduce dependence.")

## 실험 4. 토큰 치환에 대한 내성

실험 2의 같은 200편·200토큰과 바꾸지 않은 none 기준선을 재사용하고, 각 응답의 0·10·20·30·50% 위치를 다른 일반 토큰으로 치환합니다. 편집 후 문맥과 반복 마스크를 모두 다시 계산하며, 글의 각 500편·400토큰·M3 Pro보다 축소된 조건이라 값이 다릅니다. 이는 무작위 토큰 치환 실험으로 자연스러운 의역 공격과는 다릅니다.

In [ ]:
def edit_tokens(tokens, rate, allowed_ids, seed=9841):
    """Exactly floor(rate * length) changed positions per row; uniform other IDs.

    One random position ordering and replacement pool per row are shared across
    rates, making the corruption sets nested. Replacement excludes special IDs.
    """
    rng = torch.Generator().manual_seed(seed)
    tokens = tokens.clone()
    n, length = tokens.shape
    count = int(rate * length)
    ordering = torch.rand((n, length), generator=rng).argsort(-1)
    replacements = allowed_ids[torch.randint(len(allowed_ids), (n, length), generator=rng)]
    collision = replacements == tokens
    while collision.any():
        replacements[collision] = allowed_ids[torch.randint(
            len(allowed_ids), (int(collision.sum()),), generator=rng)]
        collision = replacements == tokens
    selected = ordering[:, :count]
    rows = torch.arange(n)[:, None]
    tokens[rows, selected] = replacements[rows, selected]
    return tokens


n, length, directory = N_DETECT, LENGTH, None
print_scope(False)
print("Edits: fixed seed=9841, exact floor(r*length) substitutions per response; "
      "uniform non-special vocabulary, different from original ID; no model used.")
baseline = load_data(directory, "none", n=n, length=length)
for method in ("synthid", "gumbel"):
    data = load_data(directory, method, n=n, length=length)
    ensure_comparable(baseline, data)
    null, _ = score_data(baseline, method, length, n=n)
    print(f"\n{method}, n={n}, length={length}; unchanged exp2 baseline")
    for rate in [0, 0.1, 0.2, 0.3, 0.5]:
        edited = edit_tokens(data["tokens"][:n, :length], rate, data["allowed_token_ids"])
        positive, counts = score_tokens(edited, data["key"], 30, method)
        result = metrics(null, positive)
        print(f"r={rate:.1f}: threshold={result['threshold']:.6f}, "
              f"FPR={result['fpr']:.4f}, TPR={result['tpr']:.4f}, "
              f"mean={positive.mean():.6f}, valid positions={counts.double().mean():.2f}")

## 실험 6. 엔트로피와 반복 문맥

설명형 eli5와 나열형 list를 각각 50편·200토큰 생성하여 탐지 점수, 반복 문맥 제외 비율, 기본 분포의 엔트로피를 비교합니다. 엔트로피는 온도·top-k 적용 후, 워터마크 적용 전의 nats 단위이며 반복 위치도 포함합니다. 글의 각 500편·400토큰·M3 Pro와는 다른 축소 조건이고, 나열 지시가 실제로 엔트로피를 낮추는지는 모델의 지시 이행과 생성 경로에 따라 달라집니다.

In [ ]:
for prompts in ("eli5", "list"):
    datasets[("synthid", prompts)] = engine.generate(
        method="synthid", m=30, n=N_ENTROPY, length=LENGTH,
        prompts=prompts, batch=BATCH, save_entropy=True)
n, length, directory = N_ENTROPY, LENGTH, None
print_scope(False)
print("Entropy is in nats, before watermarking, after temperature=0.7 and top-k=50; "
      "all generated positions, including masked positions.")
print("Score: average per response; mask and entropy: average per generated token.")
for prompts in ("eli5", "list"):
    data = load_data(directory, "synthid", prompts=prompts, n=n, length=length)
    if "entropy" not in data:
        raise ValueError("Missing entropy; regenerate with generate.py --save-entropy")
    scores, _ = score_data(data, "synthid", length, n=n)
    _, masked = detection_values(data["tokens"][:n, :length], data["key"], 30)
    assert (masked == data["masked"][:n, :length]).all(), "Generation/detection mask mismatch"
    entropy = data["entropy"][:n, :length]
    print(f"{prompts}: n={n}, length={length}, mean score={scores.mean():.6f}, "
          f"score SD={scores.std():.6f}, masked fraction={masked.float().mean():.6f}, "
          f"mean base entropy={entropy.mean():.6f} nats/token")
print("Handwritten list instructions are intended to lower entropy; the observed "
      "difference depends on model compliance, generated trajectories and length.")

## 실험 5. 생성 속도

배치 1·200토큰에서 각 방법을 한 번 워밍업한 뒤 다섯 번 측정하여 토큰당 시간의 중앙값·최솟값·최댓값을 비교합니다. 모델 로드는 제외하고 프롬프트 prefill, CPU 해싱·샘플링과 장치 전송은 포함하며 CUDA는 측정 전후 동기화합니다. 글의 각 500편·400토큰·M3 Pro 생성 조건과 다른 축소 실습이고 하드웨어도 달라 속도 수치를 직접 비교할 수 없습니다.

In [ ]:
length = LATENCY_LENGTH
print(f"device={engine.device}, batch=1, length={length}")
print("One warm-up per method, then 5 runs; model load excluded.")
medians = {}
for method in ("none", "synthid", "gumbel"):
    engine.generate(method=method, n=1, length=length, batch=1, progress=False)
    times = []
    for repeat in range(5):
        data = engine.generate(method=method, n=1, length=length, batch=1,
                               seed=1234 + repeat, progress=False)
        times.append(1000 * data["total_seconds"] / length)
    median = statistics.median(times)
    medians[method] = median
    increase = 100 * (median / medians["none"] - 1)
    print(f"{method}: median={median:.3f}, min={min(times):.3f}, max={max(times):.3f} "
          f"ms/token; vs none={increase:+.2f}%; runs={[round(x, 3) for x in times]}", flush=True)